# BQML と AutoML による時系列予測

**学習目標**
 1. `CREATE MODEL` を使用して、BQML で分類時系列モデルを作成する方法を学ぶ。
 2. BQML を使用して、線形回帰時系列モデルを作成する方法を学ぶ。
 3. AutoML Tables を使用して、BigQuery 内のデータから時系列モデルを構築する方法を学ぶ。

## 環境変数の設定と必要なライブラリの読み込み

In [ ]:
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]
%env PROJECT = {PROJECT}
%env REGION = "us-central1"

%load_ext google.cloud.bigquery

## データセットの作成

In [ ]:
from google.cloud import bigquery
from IPython import get_ipython

bq = bigquery.Client(project=PROJECT)


def create_dataset():
    dataset = bigquery.Dataset(bq.dataset("stock_market"))
    try:
        bq.create_dataset(dataset)  # データセットがすでに存在する場合は失敗します。
        print("Dataset created")
    except:
        print("Dataset already exists")


def create_features_table():
    error = None
    try:
        bq.query(
            """
        CREATE TABLE stock_market.eps_percent_change_sp500
        AS
        SELECT *
        FROM `stock_market.eps_percent_change_sp500`
        """
        ).to_dataframe()
    except Exception as e:
        error = str(e)
    if error is None:
        print("Table created")
    elif "Already Exists" in error:
        print("Table already exists.")
    else:
        print(error)
        raise Exception("Table was not created.")


create_dataset()
create_features_table()

## データセットの確認

前のラボでは、モデリングに使用するデータを作成し、BigQuery にテーブルとして保存しました。すべてが想定どおりであることを確認するために、そのテーブルをもう一度調べてみましょう。その後、このテーブルを使用して BigQuery ML でモデルを構築します。

In [ ]:
%%bigquery --project $PROJECT
#standardSQL
SELECT
  *
FROM
  stock_market.eps_percent_change_sp500
LIMIT
  10

## BQML の使用

### `direction` の分類モデルを作成する

モデルを作成するには:
1. `CREATE MODEL` を使用し、作成されるモデルの宛先テーブルを指定します。あるいは、既存のモデルの上書きを許可する `CREATE OR REPLACE MODEL` を使用することもできます。
2. `OPTIONS` を使用してモデルタイプ（linear_reg または logistic_reg）を指定します。正規化や学習率など、[指定できるオプション](https://cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-create#model_option_list) は他にも多数ありますが、ここではデフォルト値を受け入れます。
3. トレーニングデータを取得するクエリを指定します。

別の例については、[このチュートリアルのステップ 2](https://cloud.google.com/bigquery/docs/bigqueryml-natality) を参照してください。

**クエリの完了には約 2 分かかります**


まず、各銘柄の騰落方向（`direction`）を予測する分類モデルの作成から始めます。

`symbol` の値を使用してランダム分割を行います。約 500 種類の異なる値がある中で、`ABS(MOD(FARM_FINGERPRINT(symbol), 15)) = 1` を使用すると 30 の個別の `symbol` 値が得られ、これは約 171,000 件のトレーニング例に対応します。その 70% をトレーニング用とすると、約 110,000 件のトレーニング例でモデルを構築することになります。

In [ ]:
%%bigquery --project $PROJECT
#standardSQL
CREATE OR REPLACE MODEL
  stock_market.direction_model OPTIONS(model_type = "logistic_reg",
    input_label_cols = ["direction"]) AS
  -- トレーニングデータを取得するクエリ
SELECT
  symbol,
  Date,
  Open,
  close_MIN_prior_5_days,
  close_MIN_prior_20_days,
  close_MIN_prior_260_days,
  close_MAX_prior_5_days,
  close_MAX_prior_20_days,
  close_MAX_prior_260_days,
  close_AVG_prior_5_days,
  close_AVG_prior_20_days,
  close_AVG_prior_260_days,
  close_STDDEV_prior_5_days,
  close_STDDEV_prior_20_days,
  close_STDDEV_prior_260_days,
  direction
FROM
  `stock_market.eps_percent_change_sp500`
WHERE
  tomorrow_close IS NOT NULL
  AND ABS(MOD(FARM_FINGERPRINT(symbol), 15)) = 1
  AND ABS(MOD(FARM_FINGERPRINT(symbol), 15 * 100)) <= 15 * 70

## トレーニング統計の取得とトレーニング情報の確認

モデルを作成した後、[`ML.EVALUATE` 関数](https://cloud.google.com/bigquery-ml/docs/bigqueryml-natality#step_four_evaluate_your_model) を使用してパフォーマンスを評価できます。このコマンドを使用すると、分類モデルの適合率（precision）、再現率（recall）、正解率（accuracy）、F1 スコア、および AUC を取得できます。

In [ ]:
%%bigquery --project $PROJECT
#standardSQL
SELECT
  *
FROM
  ML.EVALUATE(MODEL `stock_market.direction_model`,
    (
    SELECT
      symbol,
      Date,
      Open,
      close_MIN_prior_5_days,
      close_MIN_prior_20_days,
      close_MIN_prior_260_days,
      close_MAX_prior_5_days,
      close_MAX_prior_20_days,
      close_MAX_prior_260_days,
      close_AVG_prior_5_days,
      close_AVG_prior_20_days,
      close_AVG_prior_260_days,
      close_STDDEV_prior_5_days,
      close_STDDEV_prior_20_days,
      close_STDDEV_prior_260_days,
      direction
    FROM
      `stock_market.eps_percent_change_sp500`
    WHERE
      tomorrow_close IS NOT NULL
      AND ABS(MOD(FARM_FINGERPRINT(symbol), 15)) = 1
      AND ABS(MOD(FARM_FINGERPRINT(symbol), 15 * 100)) > 15 * 70
      AND ABS(MOD(FARM_FINGERPRINT(symbol), 15 * 100)) <= 15 * 85))

BigQuery によって収集されたトレーニング統計を確認することもできます。トレーニング結果を表示するには、[`ML.TRAINING_INFO`](https://cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-train) 関数を使用します。

In [ ]:
%%bigquery --project $PROJECT
#standardSQL
SELECT
  *
FROM
  ML.TRAINING_INFO(MODEL `stock_market.direction_model`)
ORDER BY iteration

### 単純なベンチマークとの比較

モデルのパフォーマンスを評価するもう 1 つの方法は、単純なベンチマークと比較することです。これは、最頻クラス（多数派クラス）を単純に予測するという単純な戦略を使用してどのような精度が得られるかを確認することで行えます。トレーニングデータセットでは、最頻クラスは 'STAY' です。次のクエリで、この単純な戦略が評価セットでどのように機能するかを確認できます。

In [ ]:
%%bigquery --project $PROJECT
#standardSQL
WITH
  eval_data AS (
  SELECT
    symbol,
    Date,
    Open,
    close_MIN_prior_5_days,
    close_MIN_prior_20_days,
    close_MIN_prior_260_days,
    close_MAX_prior_5_days,
    close_MAX_prior_20_days,
    close_MAX_prior_260_days,
    close_AVG_prior_5_days,
    close_AVG_prior_20_days,
    close_AVG_prior_260_days,
    close_STDDEV_prior_5_days,
    close_STDDEV_prior_20_days,
    close_STDDEV_prior_260_days,
    direction
  FROM
    `stock_market.eps_percent_change_sp500`
  WHERE
    tomorrow_close IS NOT NULL
    AND ABS(MOD(FARM_FINGERPRINT(symbol), 15)) = 1
    AND ABS(MOD(FARM_FINGERPRINT(symbol), 15 * 100)) > 15 * 70
    AND ABS(MOD(FARM_FINGERPRINT(symbol), 15 * 100)) <= 15 * 85)
SELECT
  direction,
  (COUNT(direction)* 100 / (
    SELECT
      COUNT(*)
    FROM
      eval_data)) AS percentage
FROM
  eval_data
GROUP BY
  direction

したがって、最頻クラスを単純に推測する単純な戦略の場合、評価データセットでの精度は 0.5509 となり、BQML モデルの精度をわずかに下回ります。

### `normalized change` の回帰モデルを作成する

BigQuery を使用して、各銘柄の正規化された変化（normalized change）を予測する回帰モデルをトレーニングすることもできます。BigQuery でこれを行うには、`CREATE OR REPLACE MODEL` を呼び出す際に OPTIONS を変更するだけです。これにより、株価が上昇するか、下落するか、または横ばいかを単に予測するだけでなく、より詳細な予測が得られます。このように、ビジネスニーズに応じて、この問題を回帰問題または分類問題のいずれかとして扱うことができます。

In [ ]:
%%bigquery --project $PROJECT
#standardSQL
CREATE OR REPLACE MODEL
  stock_market.price_model OPTIONS(model_type = "linear_reg",
    input_label_cols = ["normalized_change"]) AS
  -- トレーニングデータを取得するクエリ
SELECT
  symbol,
  Date,
  Open,
  close_MIN_prior_5_days,
  close_MIN_prior_20_days,
  close_MIN_prior_260_days,
  close_MAX_prior_5_days,
  close_MAX_prior_20_days,
  close_MAX_prior_260_days,
  close_AVG_prior_5_days,
  close_AVG_prior_20_days,
  close_AVG_prior_260_days,
  close_STDDEV_prior_5_days,
  close_STDDEV_prior_20_days,
  close_STDDEV_prior_260_days,
  normalized_change
FROM
  `stock_market.eps_percent_change_sp500`
WHERE
  normalized_change IS NOT NULL
  AND ABS(MOD(FARM_FINGERPRINT(symbol), 15)) = 1
  AND ABS(MOD(FARM_FINGERPRINT(symbol), 15 * 100)) <= 15 * 70

先ほどと同様に、回帰モデルの評価指標を調べ、BigQuery でトレーニング統計を確認できます。

In [ ]:
%%bigquery --project $PROJECT
#standardSQL
SELECT
  *
FROM
  ML.EVALUATE(MODEL `stock_market.price_model`,
    (
    SELECT
      symbol,
      Date,
      Open,
      close_MIN_prior_5_days,
      close_MIN_prior_20_days,
      close_MIN_prior_260_days,
      close_MAX_prior_5_days,
      close_MAX_prior_20_days,
      close_MAX_prior_260_days,
      close_AVG_prior_5_days,
      close_AVG_prior_20_days,
      close_AVG_prior_260_days,
      close_STDDEV_prior_5_days,
      close_STDDEV_prior_20_days,
      close_STDDEV_prior_260_days,
      normalized_change
    FROM
      `stock_market.eps_percent_change_sp500`
    WHERE
      normalized_change IS NOT NULL
      AND ABS(MOD(FARM_FINGERPRINT(symbol), 15)) = 1
      AND ABS(MOD(FARM_FINGERPRINT(symbol), 15 * 100)) > 15 * 70
      AND ABS(MOD(FARM_FINGERPRINT(symbol), 15 * 100)) <= 15 * 85))

In [ ]:
%%bigquery --project $PROJECT
#standardSQL
SELECT
  *
FROM
  ML.TRAINING_INFO(MODEL `stock_market.price_model`)
ORDER BY iteration

## AutoML を使用した時系列モデルのトレーニング

### ステップ 1: データセットの作成

GCP ナビゲーションメニューで、**Agent Platform -> Datasets**（データセット）に移動します。

**Create**（作成）を選択し、`stock_market` などの名前を付けます。**Select a data type and objective**（データタイプと目標の選択）セクションで **Tabular**（表形式）を選択し、Tabular の下の **Regression/Classification**（回帰/分類）を選択します。下部の **Create**（作成）をクリックします。

<img src='../assets/create_dataset.png' width='70%'>

### ステップ 2: データのインポート

データセットを作成したら、BigQuery を使用してデータを追加できます。**Select a data source**（データソースの選択）で **Select a table or view from BigQuery**（BigQuery からテーブルまたはビューを選択）を選択します。BigQuery パスフィールドで、`eps_percent_change_sp500` を参照して指定します。

<img src='../assets/import_data.png' width='50%'>

### ステップ 3: モデルのトレーニング

データがデータセットにインポートされたら、**GENERATE STATISTICS**（統計情報を生成）をクリックして、各特徴量列の基本統計量を取得できます（注: これには数分かかります）。データがインポートされたので、分類モデルのトレーニングに進むことができます。左の列から **Train a new model**（新しいモデルをトレーニング）を選択します。目標（objective）を「Classification」（分類）に設定します。モデルのトレーニング方法が AutoML に設定されていることを確認してください。**Continue**（続行）をクリックします。

<img src='../assets/set_training_objective.png' width='80%'>

Model details（モデルの詳細）で、ターゲット列（Target Column）として `direction` 列を選択します。これは、モデルをトレーニングしたいラベル列です。**Continue**（続行）をクリックします。

<img src='../assets/select_target_label.png' width='80%'>

Training options（トレーニングオプション）で、以下**以外**のすべての列を除外します
```
symbol
Date
Open
close_MIN_prior_5_days
close_MIN_prior_20_days
close_MIN_prior_260_days
close_MAX_prior_5_days
close_MAX_prior_20_days
close_MAX_prior_260_days
close_AVG_prior_5_days
close_AVG_prior_20_days
close_AVG_prior_260_days
close_STDDEV_prior_5_days
close_STDDEV_prior_20_days
close_STDDEV_prior_260_days
```
これらは、BQML モデルで使用したものと同じ列です。これにより、BQML モデルと AutoML モデルのパフォーマンスを比較しやすくなります。
<img src='../assets/select_training_columns.png' width='80%'>

Compute and pricing（コンピューティングと料金）で、**Budget**（予算）を 1 に設定します。わずか 1 ノード時間のトレーニングで、モデルの十分なパフォーマンスが得られます。**Enable early stopping**（早期停止を有効にする）を選択しておくと、改善が見られない場合にモデルのトレーニングが確実に停止します。これで **START TRAINING**（トレーニングを開始）を選択できます。モデルのトレーニングが開始されます。

<img src='../assets/select_node_hours.png' width='80%'>

### ステップ 4: モデルの評価

トレーニングには数時間かかる場合があります。しかし、トレーニングが完了すると、モデルの評価指標を検査できます。これは分類タスクであるため、しきい値を調整して、異なるしきい値が評価指標にどのように影響するかを調べることもできます。また、そのページでは、モデルで使用されているさまざまな特徴量の重要度（feature importance）を調べたり、モデル予測の混同行列（confusion matrix）を表示したりすることもできます。

<img src='../assets/analyze_eval_metrics.png' width='80%'>

### ステップ 5: トレーニング済みモデルによる予測

モデルのトレーニングが完了したら、`Deploy and test`（デプロイとテスト）タブに移動し、モデルをエンドポイントにデプロイして予測をテストできるようにします。

<img src='../assets/deploy_model_endpoint.png' width='80%'>

予測を呼び出す際は、BigQuery テーブルまたは CSV ファイルを指定してバッチ予測ジョブを呼び出すことができます。または、単一インスタンスに対してオンライン予測を実行することもできます。

Copyright 2019 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License